# Dataset-usage detection on Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**. In the key icon on the left (Secrets), add `SHIELD_KEY` with a long random value and allow this notebook to read it. Use the same value in every session: the owner's message is derived from it, so a different key means a different message.

Everything is saved to your Google Drive, so a disconnected session loses at most 200 training steps. To continue, reopen this notebook and run the cells again: finished work is skipped and fine-tuning resumes.

In [ ]:
# 1. Is there a GPU?
!nvidia-smi -L || echo 'NO GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Keep results on Drive, and read the secret key
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['SHIELD_WORK'] = '/content/drive/MyDrive/kondor-shield/work'
os.environ['SHIELD_OUT'] = '/content/drive/MyDrive/kondor-shield/out'
os.environ['SHIELD_KEY'] = userdata.get('SHIELD_KEY')

In [ ]:
# 3. Get the code and install
%cd /content
!test -d kondor || git clone --branch dataset-usage-detection --single-branch https://github.com/0vertake/kondor.git
%cd /content/kondor
!git pull -q
!pip install -q -r requirements.txt
!python -m pytest -q tests

## Measure this GPU
Real speed for both models. Read `E2_two_finetunes_plus_sampling_hours` before committing to a long run.

In [ ]:
!python timing_test.py --model google/ddpm-celebahq-256 --batch 4 --train-steps 20 --sample-steps 20
!python timing_test.py --model google/ddpm-cifar10-32 --batch 32 --train-steps 20 --sample-steps 20

## Stage 1 and the 32 px transfer test
Trains the engine, runs E1, then the reduced E2. Needs no data of your own.

In [ ]:
!python stage1_engine.py --config px32
!python stage5_transfer.py --steps 2000 --n 500 --sample-batch 128

## The design's E2 at 256 px
Needs training images on Drive. Set `IMAGES` to a folder of images whose license allows this experiment (open decision D6). The base model is a face model, so face images are the natural choice. The patch bank needs its own images too; those may be the same folder.

In [ ]:
IMAGES = '/content/drive/MyDrive/kondor-shield/images'
!python stage1_engine.py --config px256 --photos "$IMAGES"
!python stage5_transfer.py --config px256 --images "$IMAGES" --steps 2000 --batch 4 --n 500 --sample-batch 16